In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install torchinfo

# **Import Libraries**

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import PIL
from PIL import Image
from tqdm.notebook import tqdm
import psutil
import platform

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_curve, auc, precision_recall_fscore_support)
from sklearn.preprocessing import label_binarize
from itertools import cycle

# **System Information**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# **Configuration**

In [ ]:
data_dir = "/kaggle/input/multi-cancer/Multi Cancer/Multi Cancer/Cervical Cancer"
output_dir = "/kaggle/working/processed-dataset"  
checkpoints_path = "/kaggle/working/"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(checkpoints_path, exist_ok=True)

train_batch = 32
test_batch = 16
total_class = 5 
learning_rate = 0.0001
decay = 1e-4
epoch = 100   
patience = 10


# **Data Loader**

In [13]:
def load_data(root_dir):
    file_paths = []
    labels = []
    print(f"Scanning directory: {root_dir}...")
    
    if not os.path.exists(root_dir):
        print(f"ERROR: Directory not found: {root_dir}")
        return pd.DataFrame()

    classes = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
    print(f"Detected classes: {classes}")

    for class_name in classes:
        class_dir = os.path.join(root_dir, class_name)
        for root, dirs, files in os.walk(class_dir):
            for file in files:
                if file.lower().endswith(('.bmp', '.jpg', '.png', '.jpeg')):
                    file_paths.append(os.path.join(root, file))
                    labels.append(class_name)

    df = pd.DataFrame({"file_path": file_paths, "label": labels})
    
    if not df.empty:
        df['filename_stem'] = df['file_path'].apply(lambda x: os.path.splitext(os.path.basename(x))[0])
        df = df.drop_duplicates(subset=['label', 'filename_stem']).drop(columns=['filename_stem'])

    print(f"Total images found: {len(df)}")
    return df

data = load_data(data_dir)

Scanning directory: /kaggle/input/multi-cancer/Multi Cancer/Multi Cancer/Cervical Cancer...
Detected classes: ['cervix_koc', 'cervix_dyk', 'cervix_pab', 'cervix_sfi', 'cervix_mep']
Total images found: 25000


# **Data Split and Preprocessing**